In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import plotly.express as px
import matplotlib.pyplot as plt
import bioframe as bf

pd.options.display.max_columns = 100

In [ ]:
methods_extended = ['dicast'] + methods + ['one caller support', 'two caller support', 'three caller support']
pr_rc_dict = {'method' : [], 'precision' : [], 'recall' : []}
for method in methods_extended:
    precision, recall, _ = precision_recall_curve(result['confirmed'], result['qual_' + method])
    pr_rc_dict['method'].extend([method] * (len(precision) - 1))
    pr_rc_dict['precision'].extend(precision[1:])
    pr_rc_dict['recall'].extend(recall[1:])
pr_rc_df = pd.DataFrame(pr_rc_dict)

In [ ]:
colors = ['black', '#1f77b4', '#ff7f0e', 'darkred', '#1f77b4', '#ff7f0e', 'darkred']
dash = ['solid', 'solid', 'solid', 'dot', 'dot', 'dot', 'dot']
circle_bg_white = [0, 0, 0, 1, 1, 1, 1]
fig = px.line(x='recall', y='precision', color='method', 
              data_frame=pr_rc_df, 
              title='Precision-Recall Curve without Manual Curation (' + TYPE + ')', line_dash='method', 
              line_dash_sequence=dash,
              color_discrete_sequence=colors)

for i, method in enumerate(methods_extended):
    x = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'recall']
    y = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'precision']
    if circle_bg_white[i] == 1:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor='white')
    else:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor=colors[i])

fig.update_layout(plot_bgcolor='white', xaxis_title='Recall', yaxis_title='Precision', xaxis_linecolor='black', yaxis_linecolor='black')
fig.update_traces(line=dict(width=2))
fig.update_xaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.update_yaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.write_image('figures/pr_curve_no_curation_' + TYPE + '.png', width=1200, height=650, scale=3)